# Getting Started with Docker

## What's covered

- **What a container actually is** — an isolated process tree sharing the host kernel, not a tiny VM
- **Image vs container** — the frozen template versus the running instance
- **Why Docker matters** — where containers live in production today
- **Getting Docker installed** — Docker Desktop, Docker Engine on Linux, Colima, cloud VM, with a recommended path
- **Your first container** — `docker run hello-world` and what just happened
- **The Docker architecture** — client, daemon, `containerd`, `runc`, and how they fit together
- **Anatomy of a `docker` command** — the modern `docker <object> <verb>` syntax (and the short aliases)
- **The core lifecycle** — `run`, `ps`, `stop`, `start`, `rm`, plus exit codes
- **Getting help when you forget** — `--help`, `docker help`, the docs

By the end of this notebook you should be able to install Docker, run your first container, inspect it, stop it, clean it up, and know what each of those words means. We assume you're comfortable on a Linux shell — `cd`, `ls`, `ps`, `cat` — but assume nothing about images, containers, or the daemon.

## What is a container?

If you ask ten people what a container is, you'll get ten slightly different answers. The cleanest one-line definition:

> **A container is an isolated process tree on a host, running its own filesystem and seeing its own network, while sharing the host's kernel.**

That sentence does a lot of work. Let's pick it apart.

**An isolated process tree.** When you start a container, the kernel gives it its own process namespace. Inside the container, `ps` shows only the processes that container started — PID 1 is *its* PID 1, not the host's. From the host, those same processes are still visible (with different PIDs) — the kernel is just lying to the container about who else is on the box.

**Its own filesystem.** The container sees a root filesystem (`/`) that was assembled from an **image** — a tarball of files that someone built earlier. Its `/etc`, `/usr`, `/var` are *not* your host's `/etc`, `/usr`, `/var`. They live in a separate area of disk and get mounted in at start.

**Its own network.** By default the container gets a virtual network interface with its own IP address, on a bridge network managed by Docker. Sockets it opens are *its* sockets, not your host's.

**Sharing the host's kernel.** This is the punchline. A container does **not** boot its own Linux. There's no separate kernel inside the container — every syscall the container makes goes straight to your host's kernel. That's why a container starts in milliseconds, weighs megabytes instead of gigabytes, and can't run a different OS than the host kernel.

### Containers vs virtual machines

The picture that locks this in:

```
       Virtual Machines                       Containers
   +------+ +------+ +------+             +------+ +------+ +------+
   | App  | | App  | | App  |             | App  | | App  | | App  |
   +------+ +------+ +------+             +------+ +------+ +------+
   | libs | | libs | | libs |             | libs | | libs | | libs |
   +------+ +------+ +------+             +------+ +------+ +------+
   |Guest | |Guest | |Guest |             +------------------------+
   |  OS  | |  OS  | |  OS  |             |   Docker Engine        |
   +------+-+------+-+------+             +------------------------+
   |      Hypervisor        |             |   Host Linux kernel    |
   +------------------------+             +------------------------+
   |   Host OS / hardware   |             |       Hardware         |
   +------------------------+             +------------------------+
```

A VM virtualises the **hardware**, so each guest brings its own kernel, drivers, and OS — heavy but very strongly isolated. A container virtualises the **operating system**, so each container is just a bag of processes the kernel keeps separate — light and fast, but isolation is "kernel-strong," not "hypervisor-strong."

### The kernel features under the hood

Docker doesn't invent isolation. It composes Linux features that have existed for years:

- **Namespaces** — give each container its own view of PIDs, mounts, network, users, hostnames, and IPC.
- **Control groups (cgroups)** — meter and limit how much CPU, memory, I/O, and pids a container can consume.
- **Capabilities** — slice the all-powerful root user into ~40 fine-grained privileges, so a container's root isn't really *your* root.
- **Seccomp, AppArmor, SELinux** — restrict which syscalls and operations the container is allowed to make.

We'll meet each of these properly in later notebooks. For now, just know: **Docker is the friendly front end on top of kernel primitives that were already there.**

## Image vs container

Two words you'll use every day, and they mean different things.

An **image** is a frozen, read-only template — a snapshot of a filesystem plus a tiny bit of metadata (what command to run, what user, what environment variables). It's like a class in object-oriented programming. It doesn't *do* anything; it just sits there as a blueprint.

A **container** is a running (or stopped) instance of an image. It's like an object: created from the class, with its own state. You can create many containers from the same image, and they each get their own writable layer on top of the image's read-only ones.

A useful pair of analogies:

| Concept | Image | Container |
|---|---|---|
| OOP | Class | Object |
| Cooking | Recipe | The actual dish you cooked |
| Software | The installer | The installed, running program |

You **build** (or **pull**) an image. You **run** an image to create a container. You **stop**, **start**, and **remove** containers. Notebook 02 goes deep on images; notebook 03 goes deep on containers. For now, just remember which word means which.

## Why Docker matters

If you work near software in 2026, you're standing on containers whether you know it or not.

- **Every cloud runs them.** AWS ECS/Fargate/EKS, Google Cloud Run/GKE, Azure ACI/AKS — all are container platforms. When you "deploy a service" in any modern cloud, you're shipping a container image.
- **Kubernetes pods are containers.** Kubernetes doesn't replace Docker; it orchestrates containers built and packaged the way Docker established. Learning Docker is a prerequisite to Kubernetes.
- **CI/CD runs on containers.** GitHub Actions, GitLab CI, CircleCI — each step usually runs inside a container so builds are reproducible across machines.
- **Local development.** `docker compose up` to bring up a Postgres + Redis + your service for local work has replaced "install five things on your laptop" for most teams.
- **Reproducibility.** "Works on my machine" mostly died when teams started shipping the machine.

The **DCA** (Docker Certified Associate) exam is the most widely-recognised Docker certification, and this course is structured to take you from zero through to ready-to-sit. By notebook 10 you'll have hands-on familiarity with every domain the exam tests.

## Getting Docker installed

You only learn Docker by running containers. Here are the four common ways to get a working Docker on your machine, ordered roughly by ease.

**Option 1 — Docker Desktop (recommended for laptops).** Free for personal use; paid for larger companies. Available for macOS, Windows, and Linux. Install the `.dmg`/`.exe`/`.deb`, launch the app, and you have `docker` on your `PATH`.

- Pros: one-click install, GUI for images/containers/volumes, includes Docker Compose, Kubernetes toggle.
- Cons: licensing for big orgs; on macOS/Windows it secretly runs a tiny Linux VM (the "Docker VM") because containers need a Linux kernel — so file-sharing between host and container can be slow.

**Option 2 — Docker Engine on Linux (the real thing).** If you have a Linux box (or a cloud VM), install Docker Engine directly. Native, no VM, the fastest experience.

```bash
# Ubuntu/Debian (abbreviated; see notebook 10 for the full hardened install)
curl -fsSL https://get.docker.com | sh
sudo usermod -aG docker $USER
newgrp docker
```

- Pros: native performance, what production uses, no licensing question.
- Cons: needs a Linux host; harder if you're on a Mac/Windows laptop.

**Option 3 — Colima (open-source Docker Desktop alternative).** macOS/Linux. Spins up a small Lima VM and runs Docker Engine inside it. Free, no licensing, very Mac-friendly.

```bash
brew install colima docker
colima start
```

- Pros: free for everyone, lightweight, no GUI overhead.
- Cons: CLI only, slightly more setup than Docker Desktop.

**Option 4 — Cloud VM with Docker Engine.** Spin up a small Linux VM (DigitalOcean droplet, AWS EC2 t3.small, Hetzner CX11) and install Docker Engine. SSH in to use it.

- Pros: real native Linux Docker, mirrors production, can break and rebuild freely.
- Cons: costs a few dollars a month, needs internet, slight friction to use day-to-day.

**For this course**, Docker Desktop or Colima is the path of least friction if you're on a laptop. A cloud VM is the most authentic experience. Pick one and stick with it.

### Verify the install

Once Docker is installed, two commands tell you it's healthy:

- `docker version` — shows the client and server versions. If you see *both*, the daemon is running and you can talk to it.
- `docker info` — shows the daemon's configuration: storage driver, logging driver, kernel version, number of running/stopped containers, default registry.

In [ ]:
!docker version
!docker info | head -25

## Your first container — `docker run hello-world`

The Docker equivalent of "hello world":

```bash
docker run hello-world
```

Run that. You'll see something like:

```
Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world
...
Status: Downloaded newer image for hello-world:latest

Hello from Docker!
This message shows that your installation appears to be working correctly.
...
```

That output is dense with information. Let's read it line by line, because every container you ever run goes through this same sequence.

1. **`Unable to find image 'hello-world:latest' locally`** — the daemon checked your local image cache for an image named `hello-world` with the tag `latest`. It wasn't there.
2. **`Pulling from library/hello-world`** — so the daemon went out to **Docker Hub** (the default registry) and downloaded it. `library/` is the namespace for official images.
3. **`Status: Downloaded newer image for hello-world:latest`** — the image now lives in your local cache. Next time you `docker run hello-world`, this step is skipped.
4. **The "Hello from Docker!" text** — the daemon then *created a container* from that image, *started it*, and the container printed its message and exited.

So `docker run hello-world` is shorthand for:

```
pull (if needed) -> create container -> start container -> stream output -> container exits
```

Every `docker run` is some variation on those steps.

In [ ]:
!docker run hello-world

### What got created?

The container exited, but it's not gone — Docker keeps stopped containers around so you can inspect them and start them again. Two commands prove it:

- `docker ps` — list **running** containers. After `hello-world` exits, this shows nothing.
- `docker ps -a` — list **all** containers, including stopped ones. You'll see your `hello-world` container with status `Exited`.
- `docker images` — list local images. You'll see `hello-world:latest` there now.

We'll learn to clean these up at the end of this notebook.

In [ ]:
!docker ps
!echo '---'
!docker ps -a | head -5
!echo '---'
!docker images | head -5

## How Docker actually works — the architecture

When you typed `docker run hello-world`, you used the **Docker CLI** — a small client program. But the CLI didn't run the container itself. It sent a request over a Unix socket to a long-running daemon. That daemon is what actually does the work.

```
+-----------------+    REST API     +----------------------+
| docker CLI      | --------------> | dockerd (daemon)     |
| (your terminal) | /var/run/       |  - manages images    |
|                 |  docker.sock    |  - manages networks  |
+-----------------+                 |  - manages volumes   |
                                    +----------+-----------+
                                               | gRPC
                                               v
                                    +----------------------+
                                    | containerd           |
                                    |  - container runtime |
                                    |    supervisor        |
                                    +----------+-----------+
                                               | exec
                                               v
                                    +----------------------+
                                    | runc                 |
                                    |  - actually creates  |
                                    |    namespaces +      |
                                    |    cgroups, starts   |
                                    |    the process       |
                                    +----------------------+
```

Four moving parts:

- **`docker`** — the CLI you type. Stateless. Just packages your command as an HTTP request.
- **`dockerd`** — the Docker daemon. The brain. Owns the image cache, the container metadata, networks, volumes. Listens on `/var/run/docker.sock` by default.
- **`containerd`** — the container *runtime supervisor*. Tracks the state of containers, handles image pulling at the low level, talks to `runc` to actually start things. Donated by Docker to the CNCF — used by Kubernetes too.
- **`runc`** — the thing that **actually** creates the namespaces, mounts the rootfs, sets up cgroups, and `exec`s your container's process. Tiny, OCI-conformant, also a CNCF project.

Why does this matter? Two reasons.

First, when something goes wrong, you'll often need to know *which* layer to look at. "The daemon isn't running" is `dockerd`. "A container is stuck" might be `containerd` or `runc`. Notebook 10 covers the troubleshooting flow.

Second, the **client/daemon separation** means you can point your local `docker` CLI at a *remote* daemon — e.g. on a server — by setting the `DOCKER_HOST` environment variable. That's how CI systems run builds on beefier hosts.

In [ ]:
# The Docker socket is where the CLI talks to the daemon
!ls -la /var/run/docker.sock 2>/dev/null || echo 'No socket here - Docker may not be installed or running'
!echo '---'
# The daemon process itself
!ps -ef | grep -E 'dockerd|containerd' | grep -v grep | head -5 || true

## Anatomy of a `docker` command

Modern Docker organises commands by **object** then **verb**:

```
docker <object>   <verb>   [flags]   [arguments]
        |           |
        |           +-- what to do
        +-- what kind of thing (container, image, volume, network, ...)
```

Examples:

- `docker container ls` — list containers
- `docker container run nginx` — run a container from the `nginx` image
- `docker image pull ubuntu:24.04` — pull an image
- `docker image ls` — list images
- `docker volume create data` — create a volume
- `docker network ls` — list networks

The full set of objects: `container`, `image`, `volume`, `network`, `system`, `builder`, `buildx`, `compose`, `context`, `plugin`, `secret`, `service`, `stack`, `swarm`, `trust`. You'll meet each one in later notebooks.

### The short aliases (also called "legacy top-level commands")

For the most-used objects, Docker keeps short top-level aliases. These do the same thing and are what you'll see in most blog posts:

| Long form | Short alias |
|---|---|
| `docker container run` | `docker run` |
| `docker container ls` | `docker ps` |
| `docker container stop` | `docker stop` |
| `docker container rm` | `docker rm` |
| `docker image ls` | `docker images` |
| `docker image pull` | `docker pull` |
| `docker image build` | `docker build` |
| `docker image rm` | `docker rmi` |

Use whichever feels natural. The long form is more discoverable (`docker container --help` lists everything you can do to a container); the short form is faster to type. We'll mix both in this course.

## The core container lifecycle

A container has a small handful of states, and a handful of commands to move it between them.

```
                docker run
   +----------------------------------+
   |                                  v
   |       docker create        +-----------+ docker start  +-----------+
   |  image ------------------> |  created  | ------------> |  running  |
   |                            +-----------+               +-----+-----+
   |                                                              |
   |                                              docker stop     |
   |                                              (SIGTERM, then  |
   |                                               SIGKILL)       |
   |                                                              v
   |                                                        +-----------+
   |                                     docker start <---  |  stopped  |
   |                                                        +-----+-----+
   |                                                              |
   |                                                              | docker rm
   |                                                              v
   |                                                        +-----------+
   |                                                        |   gone    |
   |                                                        +-----------+
   v
```

The everyday commands:

- **`docker run IMAGE`** — create + start in one step. The 90% case.
- **`docker ps`** — list running containers. Add `-a` to include stopped ones. Add `-q` to print just IDs (useful in pipelines: `docker rm $(docker ps -aq)`).
- **`docker stop NAME`** — politely stop a container. Sends `SIGTERM`, waits 10 seconds, then `SIGKILL`. Use `--time` to change the grace period.
- **`docker start NAME`** — start a stopped container again. Keeps its writable layer, so any files it wrote are still there.
- **`docker restart NAME`** — stop then start, in one command.
- **`docker kill NAME`** — immediate `SIGKILL`, no grace period. The "I mean it" button.
- **`docker rm NAME`** — delete a stopped container. Add `-f` to force-remove a running one.
- **`docker logs NAME`** — show what the container wrote to stdout/stderr. Add `-f` to follow (like `tail -f`).

Containers can be referred to by **name** (`--name web` when you `run`, or the auto-generated one like `frosty_einstein`) or by **ID** (the hex string — first few characters are usually enough to be unique). Pick names when you'll talk to the container often; let Docker auto-name throwaways.

In [ ]:
# Run a tiny container in the background, give it a name
!docker run -d --name hello-nginx -p 8088:80 nginx:alpine
!echo '---'
# Show running containers
!docker ps
!echo '---'
# Hit it from the host
!curl -s -o /dev/null -w 'HTTP %{http_code}\n' http://localhost:8088 2>/dev/null || echo 'curl not available or container not yet ready'

In [ ]:
# Read its logs
!docker logs hello-nginx | tail -5
!echo '---'
# Stop and remove it
!docker stop hello-nginx
!docker rm hello-nginx
!docker ps -a | grep hello-nginx || echo 'container is gone'

### Important flags for `docker run`

You'll see these constantly. We'll deepen them in notebook 03, but meet them now:

- **`-d`** ("detached") — run in the background. Without it, the container takes over your terminal until it exits.
- **`--name <name>`** — assign a stable name. Without it, Docker auto-generates one like `priceless_borg`.
- **`-p <host>:<container>`** ("publish") — map a host port to a container port. `-p 8080:80` exposes the container's port 80 as host port 8080. We unpack this in notebook 05.
- **`-v <volume-or-path>:<container-path>`** ("volume") — mount a volume or host directory into the container. Notebook 04.
- **`-e KEY=value`** ("env") — set an environment variable inside the container.
- **`--rm`** — auto-remove the container when it exits. Great for short-lived tasks so you don't accumulate `Exited` containers.
- **`-it`** — combine `-i` (keep stdin open) and `-t` (allocate a TTY). The standard combo for *interactive* containers like `docker run -it ubuntu bash`.
- **`--restart <policy>`** — what to do when the container exits. `no` (default), `on-failure`, `always`, `unless-stopped`.

Try an interactive shell in an Ubuntu container:

```bash
docker run -it --rm ubuntu:24.04 bash
# inside:
cat /etc/os-release
exit
```

`--rm` cleans it up when you `exit`. Without `--rm`, you'd accumulate dead Ubuntu containers.

In [ ]:
# A short-lived task: run a one-shot command in an Alpine container, auto-clean
!docker run --rm alpine:3.20 echo 'hello from inside the container'
!echo '---'
# Confirm nothing was left behind
!docker ps -a | grep alpine || echo 'no leftover alpine containers'

## Container exit codes

When a container exits, it has an exit code — the standard Unix convention. `0` means success; anything else is a failure. You can see it in `docker ps -a` under `STATUS` (e.g. `Exited (137) 5 seconds ago`) or pull it out explicitly.

A few exit codes you'll see often:

| Code | Meaning |
|---|---|
| **0** | Clean exit, success. |
| **1** | Generic application error. |
| **125** | The `docker run` command itself failed (e.g. bad flag) — the container never started. |
| **126** | The container command was found but isn't executable. |
| **127** | The container command wasn't found at all. |
| **130** | Process killed by `SIGINT` (Ctrl-C, signal 2 -> 128+2). |
| **137** | Process killed by `SIGKILL` (signal 9 -> 128+9). Often means out-of-memory or `docker kill`. |
| **143** | Process killed by `SIGTERM` (signal 15 -> 128+15). What a clean `docker stop` looks like to the process. |

The `128 + signal` convention is Linux, not Docker — but Docker surfaces it faithfully, so it's worth memorising the `137` (OOM/kill) and `143` (graceful stop) sightings.

In [ ]:
# Demonstrate exit codes
!docker run --rm alpine:3.20 sh -c 'exit 0' && echo 'first run succeeded'
!docker run --rm alpine:3.20 sh -c 'exit 42'; echo "second run exit was $?" 

## Getting help — `--help`, `docker help`, and the docs

You won't memorise every flag of every command. You don't need to — Docker has excellent built-in help.

- **`docker --help`** — top-level command list.
- **`docker <object> --help`** — list every verb for that object. E.g. `docker container --help`, `docker image --help`.
- **`docker <object> <verb> --help`** — every flag of a specific command. E.g. `docker run --help` is the single most-read help page in Docker.
- **`docker help <command>`** — same as `--help`, slightly older spelling.

For deeper docs:

- **[docs.docker.com](https://docs.docker.com)** — the official reference. The CLI section is searchable and current.
- **Image-specific docs** — every official image on Docker Hub has a README explaining its tags, env vars, and volumes. `hub.docker.com/_/postgres` is a model example.
- **`man docker`** — on Linux installs, manpages are sometimes available too.

The flow when you're stuck: `<command> --help` first (instant, you'll find the flag), then the docs site for tutorials and conceptual explanations.

In [ ]:
!docker --help | head -25
!echo '---'
!docker run --help | head -20

## Cleaning up

A few hours into Docker and you'll have dozens of `Exited` containers and old images sitting around. Two ways to clean:

- **Targeted** — `docker rm <name-or-id>` for one container, `docker rmi <image>` for one image.
- **Bulk** — `docker system prune` removes all stopped containers, dangling images (untagged orphans), and unused networks in one go. Add `-a` to also remove all unused images (not just dangling). Add `--volumes` to also nuke volumes (careful — that's data).

```bash
docker system prune          # safe-ish: stopped containers, dangling images, unused networks
docker system prune -a       # more aggressive: also removes ALL unused images
docker system prune -a --volumes   # nuclear: also wipes unused volumes (data loss risk)
```

And to see what's taking space:

```bash
docker system df    # like `df` but for Docker - images, containers, volumes
```

In [ ]:
!docker system df
!echo '---'
!docker ps -a --format 'table {{.ID}}\t{{.Image}}\t{{.Status}}\t{{.Names}}' | head -10